In [ ]:
# Conv Architectures
layers1 = [{"c_in": 3, "d_in": 32, "c_out": 128, "d_out": 32, "k": 3}, {"c_in": 128, "d_in": 32, "c_out": 256, "d_out": 32, "k": 3}, {"c_in": 256, "d_in": 16, "c_out": 256, "d_out": 16, "k": 3}, {"c_in": 256, "d_in": 16, "c_out": 512, "d_out": 16, "k": 3}, {"c_in": 512, "d_in": 8, "c_out": 512, "d_out": 8, "k": 3}, {"c_in": 512, "d_in": 4, "c_out": 512, "d_out": 4, "k": 3}]
layers2 = [{"d_in": 32, "d_out": 32, "c_in": 3, "c_out": 128, "k": 3}, {"d_in": 32, "d_out": 32, "c_in": 128, "c_out": 128, "k": 3}, {"d_in": 32, "d_out": 32, "c_in": 128, "c_out": 128, "k": 3}, {"d_in": 32, "d_out": 32, "c_in": 128, "c_out": 256, "k": 3}, {"d_in": 16, "d_out": 16, "c_in": 256, "c_out": 256, "k": 3}, {"d_in": 16, "d_out": 16, "c_in": 256, "c_out": 512, "k": 3}, {"d_in": 8, "d_out": 8, "c_in": 512, "c_out": 512, "k": 3}, {"d_in": 8, "d_out": 8, "c_in": 512, "c_out": 512, "k": 3}, {"d_in": 4, "d_out": 4, "c_in": 512, "c_out": 512, "k": 3}]

In [2]:
# Per batch
bs=64
input_dim = 9216
dataset_size = 64

In [ ]:
# Config
bs = bs                    # batch size
optimizer = "adam"         # "sgd" or "adam"

for i, net in enumerate([layers1, layers2]):
    print("Net", i+1)

    # ---- Ops ----
    batch_mul, batch_add = 0, 0   # per batch (fwd + bwd over bs)
    upd_mul, upd_add = 0, 0       # per-step (optimizer update once)

    # ---- Memory (elements, not bytes) ----
    mem_w = 0                     # parameters (Σ per layer)
    mem_h = 0                     # saved activations (Σ per layer × bs)
    mem_d_peak = 0                # gradients (MAX if streaming else SUM)
    mem_opt = 0                   # optimizer state (persistent)

    for layer in net:
        Cin  = layer["c_in"]
        Din  = layer["d_in"]
        Cout = layer["c_out"]
        Dout = layer["d_out"]
        k    = layer.get("k", 1)

        K   = Cin * k * k
        S   = Dout * Dout
        Sin = Din * Din

        # ---- per-sample ops ----
        # forward
        mul_fwd = Cout * S * K
        add_fwd = Cout * S * (K - 1)

        # weight gradients dW
        mul_dW = Cout * Cin * k * k * S
        add_dW = Cout * Cin * k * k * (S - 1)

        # input gradients dX
        mul_dX = Cin * Sin * Cout * k * k
        add_dX = Cin * Sin * (Cout * k * k - 1)

        # ---- per-batch totals ----
        batch_mul += bs * (mul_fwd + mul_dW + mul_dX)
        batch_add += bs * (add_fwd + add_dX) + (Cout * Cin * k * k) * (bs * S - 1) # use the exact per-layer dW add count over the whole batch: (bs*S - 1) per weight

        # ---- params / grads / activations ----
        params = Cout * Cin * k * k
        mem_w += params
        mem_h += bs * Cout * S
        mem_d_peak = max(mem_d_peak, params)     # keep only largest layer's grads

        # ---- per-step optimizer update (once per step) ----
        if optimizer == "sgd":
            # 1 mul + 1 add per param (w -= lr * g)
            upd_mul += params
            upd_add += params
            opt_state_mult = 0                        # SGD w/out momentum
        elif optimizer == "adam":
            # Approximate mul/add counts per param (elementwise):
            # m = b1*m + (1-b1)*g        -> 2 mul + 1 add
            # v = b2*v + (1-b2)*g^2      -> 3 mul + 1 add  (g^2 counts as mul)
            # bias correction (m_hat, v_hat) -> 2 mul       (use precomputed inverses)
            # w -= lr * m_hat / sqrt(v_hat+eps) -> 2 mul + 1 add (ignore sqrt/div as non-mul/add)
            per_param_mul = 9
            per_param_add = 3 + 1  # +1 for the final weight add/sub
            upd_mul += params * per_param_mul
            upd_add += params * per_param_add
            opt_state_mult = 2                      # m and v
        else:
            raise ValueError("optimizer must be 'sgd' or 'adam'")

        mem_opt += params * opt_state_mult          # persistent optimizer state

    # ---- Per-step totals ----
    step_mul = batch_mul + upd_mul
    step_add = batch_add + upd_add

    print(
        f"step_mul={step_mul:,}, step_add={step_add:,}"
    )
    print(
        f"mem_w={mem_w:,} elems, mem_h={mem_h:,} elems, "
        f"mem_d={mem_d_peak:,} elems, mem_opt={mem_opt:,} elems"
    )

Net 1
step_mul=20,214,943,104, step_add=20,150,453,376
mem_w=754,048 elems, mem_h=40,370,176 elems, mem_d=262,144 elems, mem_opt=1,508,096 elems
Net 2
step_mul=29,881,273,728, step_add=29,777,265,792
mem_w=1,048,960 elems, mem_h=59,244,544 elems, mem_d=262,144 elems, mem_opt=2,097,920 elems
